In [21]:
import pandas as pd

df_cb = pd.read_csv("./data_callback/dataThuyDien/callback_SongBaHa.csv")
df_hb = pd.read_csv("./data_callback/dataThoiTiet/BaHa.csv")

In [22]:
# =========================================================
# 2) Parse thời gian robust cho callback (MM/DD/YYYY + AM/PM)
#    + clean ký tự ẩn để tránh NaT
# =========================================================
df_cb["thoi_diem_raw"] = df_cb["thoi_diem"]

s = df_cb["thoi_diem_raw"].astype(str)
s = (s
     .str.replace("\xa0", " ", regex=False)   # NBSP
     .str.replace("\u200b", "", regex=False)  # zero-width
     .str.replace("\t", " ", regex=False)
     .str.replace(r"\s+", " ", regex=True)
     .str.strip()
)
df_cb["thoi_diem_clean"] = s

# Thử nhiều format
dt1 = pd.to_datetime(df_cb["thoi_diem_clean"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
mask_fail = dt1.isna()
dt2 = pd.to_datetime(df_cb.loc[mask_fail, "thoi_diem_clean"], format="%m/%d/%Y %I:%M %p", errors="coerce")

mask_fail2 = mask_fail.copy()
mask_fail2.loc[mask_fail] = dt2.isna()
dt3 = pd.to_datetime(df_cb.loc[mask_fail2, "thoi_diem_clean"], errors="coerce")  # fallback linh hoạt

df_cb["thoi_diem"] = dt1
df_cb.loc[mask_fail, "thoi_diem"] = dt2.values
df_cb.loc[mask_fail2, "thoi_diem"] = dt3.values

# =========================================================
# 3) Parse thời gian cho BaHa (ISO)
# =========================================================
df_hb["thoi_diem"] = pd.to_datetime(df_hb["time"], errors="coerce")

# Bỏ timezone nếu có
for _df in [df_cb, df_hb]:
    try:
        _df["thoi_diem"] = _df["thoi_diem"].dt.tz_localize(None)
    except Exception:
        pass

# =========================================================
# 4) Kiểm tra NaT sau parse
# =========================================================
nat_cb = df_cb["thoi_diem"].isna()
nat_hb = df_hb["thoi_diem"].isna()

print("callback_BaHa: NaT count =", int(nat_cb.sum()), "| NaT % =", float(nat_cb.mean()))
print("BaHa        : NaT count =", int(nat_hb.sum()), "| NaT % =", float(nat_hb.mean()))

if nat_cb.any():
    print("\n--- callback_BaHa: các giá trị vẫn lỗi parse (hiển thị 30 dòng) ---")
    print(df_cb.loc[nat_cb, ["thoi_diem_raw", "thoi_diem_clean"]].head(30).to_string(index=False))

if nat_hb.any():
    print("\n--- BaHa: các giá trị time lỗi parse (hiển thị 30 dòng) ---")
    print(df_hb.loc[nat_hb, ["time"]].head(30).to_string(index=False))

# Nếu vẫn còn NaT thì dừng để tránh merge sai
if nat_cb.any() or nat_hb.any():
    raise ValueError("Vẫn còn NaT sau parse. Hãy kiểm tra các dòng in ra ở trên.")

# =========================================================
# 5) (Khuyến nghị) Loại trùng timestamp trong từng file
# =========================================================
df_cb = df_cb.sort_values("thoi_diem").drop_duplicates("thoi_diem", keep="last")
df_hb = df_hb.sort_values("thoi_diem").drop_duplicates("thoi_diem", keep="last")

# =========================================================
# 6) Merge theo thời gian
#    Khuyến nghị: BaHa là MASTER, callback là bổ sung -> how="left"
# =========================================================
df_merged = df_hb.merge(
    df_cb.drop(columns=["thoi_diem_raw", "thoi_diem_clean"], errors="ignore"),
    on="thoi_diem",
    how="left",
    suffixes=("_hb", "_cb")
)

# =========================================================
# 7) Kiểm tra sau merge
# =========================================================
print("\nMerged shape:", df_merged.shape)
print("NaT thoi_diem:", int(df_merged["thoi_diem"].isna().sum()))
print("Duplicate timestamps:", int(df_merged["thoi_diem"].duplicated().sum()))

# =========================================================
# 8) Chuẩn hoá ISO để export (tuỳ chọn)
# =========================================================
df_merged["thoi_diem_iso"] = df_merged["thoi_diem"].dt.strftime("%Y-%m-%dT%H:%M:%S")

# =========================================================
# 9) Lưu file kết quả
# =========================================================
out_path = "./data_callback/BaHa_enriched.csv"
df_merged.to_csv(out_path, index=False, encoding="utf-8")
print("Saved:", out_path)

callback_BaHa: NaT count = 0 | NaT % = 0.0
BaHa        : NaT count = 0 | NaT % = 0.0

Merged shape: (34344, 15)
NaT thoi_diem: 0
Duplicate timestamps: 0
Saved: ./data_callback/BaHa_enriched.csv
